# LeetCode #1109 — Corporate Flight Bookings

**Bài Medium · Python 3 · Difference Array (mảng hiệu) + Prefix Sum (tổng tiền tố)**  
Đề chính thức: https://leetcode.com/problems/corporate-flight-bookings/

```python
class Solution:
    def corpFlightBookings(self, bookings: List[List[int]], n: int) -> List[int]:
        ...
```

Notebook này đưa bạn từ cách cộng ghế cho từng chuyến bay `O(m*n)` đến kỹ thuật cập nhật **cả một đoạn** trong `O(1)` cho mỗi booking, rồi khôi phục kết quả bằng tổng tiền tố.

## 1. Mục tiêu học

Sau bài này, bạn có thể:
1. Phân biệt **số hiệu chuyến bay 1-based** và **chỉ số list Python 0-based**.
2. Viết lời giải đơn giản cộng từng chuyến và thấy nút thắt hiệu năng.
3. Hiểu `diff` lưu **mức thay đổi ở ranh giới**, không lưu số ghế trực tiếp.
4. Hiểu vì sao một booking `[first, last, seats]` chỉ cần `+seats` tại đầu và `−seats` **ngay sau cuối**.
5. Dùng tổng tiền tố để khôi phục số ghế; chứng minh đúng, kiểm thử và làm 5 bài tập.

## 2. Đề bài — đặt chỗ theo khoảng số hiệu

Có `n` chuyến bay được đánh số **1, 2, ..., n**.

Mỗi booking `[first, last, seats]` nghĩa là đặt `seats` ghế cho **mỗi chuyến** từ số `first` đến số `last`, **bao gồm cả hai đầu**.

Trả về list `answer` có `n` phần tử: `answer[i]` là số ghế đã đặt cho chuyến bay có số hiệu **`i+1`**.

**Ví dụ 1:**
```python
bookings = [[1,2,10], [2,3,20], [2,5,25]]
n = 5
# expected = [10,55,45,25,25]
```
**Ví dụ 2:** `bookings=[[1,2,10],[2,2,15]], n=2` → `[10,25]`.

**Ràng buộc:** `1 <= n <= 2*10**4`; `1 <= len(bookings) <= 2*10**4`; `1 <= first <= last <= n`; `1 <= seats <= 10**4`.

## 3. Tính ví dụ bằng tay

| Chuyến | Booking 1 | Booking 2 | Booking 3 | Tổng |
|---:|---:|---:|---:|---:|
| 1 | 10 | 0 | 0 | **10** |
| 2 | 10 | 20 | 25 | **55** |
| 3 | 0 | 20 | 25 | **45** |
| 4 | 0 | 0 | 25 | **25** |
| 5 | 0 | 0 | 25 | **25** |

Bạn thấy chuyến 2 bị cộng **cả ba booking** vì ba đoạn đều chứa chuyến 2. Đề hỏi **tổng ghế của tất cả booking**, không phải chọn booking lớn nhất.

In [1]:
bookings_example = [[1,2,10], [2,3,20], [2,5,25]]
flight_count_example = 5
contributions = []
for first, last, seats in bookings_example:
    row = [seats if first <= flight <= last else 0
           for flight in range(1, flight_count_example + 1)]
    contributions.append(row)
    print(f'Booking [{first},{last},{seats}]: {row}')
print('Tổng:', [sum(row[i] for row in contributions)
                for i in range(flight_count_example)])

Booking [1,2,10]: [10, 10, 0, 0, 0]
Booking [2,3,20]: [0, 20, 20, 0, 0]
Booking [2,5,25]: [0, 25, 25, 25, 25]
Tổng: [10, 55, 45, 25, 25]


## 4. Kiến thức Python — index bắt đầu từ 0

Tại sao có `first - 1` trong lời giải?

- Chuyến bay số `1` nằm ở `answer[0]`.
- Chuyến bay số `2` nằm ở `answer[1]`.
- Chuyến bay số `n` nằm ở `answer[n-1]`.

Vì vậy khi cập nhật *list Python*, phải đổi số hiệu `flight` thành index `flight-1`.

In [2]:
answer = [0] * 5
flight = 2
answer[flight - 1] += 10
print('Sau khi thêm 10 ghế cho chuyến 2:', answer)
assert answer == [0,10,0,0,0]

Sau khi thêm 10 ghế cho chuyến 2: [0, 10, 0, 0, 0]


### 4.1. Cú pháp unpacking và vòng lặp `range`

`for first, last, seats in bookings` lấy ba trường ở mỗi booking.

`range(first, last+1)` đi qua **cả `last`**; nếu quên `+1`, bạn bỏ sót chuyến cuối của đoạn.

In [3]:
first, last, seats = [2, 4, 7]
print('Các chuyến cần cập nhật:', list(range(first, last+1)))
print('Các chuyến bị bỏ sót nếu sai:', list(range(first, last)))

Các chuyến cần cập nhật: [2, 3, 4]
Các chuyến bị bỏ sót nếu sai: [2, 3]


## 5. Cách trực giác (Brute Force)

Đặt list `answer=[0]*n`. Với mỗi booking, duyệt tất cả chuyến bay trong đoạn và cộng `seats`.

Cách này **dễ hiểu và đúng**, nên dùng làm baseline để kiểm tra cách tối ưu.

In [4]:
def corp_flight_bookings_bruteforce(bookings, n):
    ans = [0] * n
    for first, last, seats in bookings:
        for flight in range(first, last + 1):
            ans[flight - 1] += seats
    return ans

assert corp_flight_bookings_bruteforce(bookings_example, 5) == [10,55,45,25,25]
print('Brute force:', corp_flight_bookings_bruteforce(bookings_example, 5))

Brute force: [10, 55, 45, 25, 25]


### 5.1. Điểm yếu về hiệu suất

Gọi `m = len(bookings)`. Trong trường hợp một booking bao phủ **toàn bộ `n` chuyến**, mỗi booking cần `n` phép cộng.

Tệ nhất: `m * n` phép cộng. Với `m=n=20.000`, đó là **400 triệu lượt cập nhật**, quá chậm trong Python.

Ta cần tìm cách diễn tả thay đổi trên cả một đoạn mà **không đi qua từng vị trí**.

## 6. Từ mảng thường đến *mảng hiệu* (difference array)

Giả sử `a=[10,10,30,30,20]` biểu diễn số ghế trên 5 chuyến bay.

Thay vì lưu `a` trực tiếp, ta lưu **chênh lệch** giữa hai vị trí liên tiếp:

- `diff[0] = a[0]`.
- `diff[i] = a[i] - a[i-1]` khi `i>=1`.

Suy ra `diff=[10,0,20,0,-10]`.

Nghĩa là: bắt đầu 10 ghế; giữ nguyên ở vị trí 2; tăng 20 tại vị trí 3; giữ nguyên; giảm 10 tại vị trí 5.

In [5]:
a = [10, 10, 30, 30, 20]
diff_from_a = [a[0]] + [a[i] - a[i-1] for i in range(1, len(a))]
print('a   :', a)
print('diff:', diff_from_a)
assert diff_from_a == [10, 0, 20, 0, -10]

a   : [10, 10, 30, 30, 20]
diff: [10, 0, 20, 0, -10]


## 7. Prefix Sum (tổng tiền tố) — phép toán đảo của mảng hiệu

Tổng tiền tố tại vị trí `i` là tổng của tất cả phần tử từ `diff[0]` đến `diff[i]`:

`a[i] = diff[0] + diff[1] + ... + diff[i]`.

Ví dụ với `diff=[10,0,20,0,-10]`:

- Vị trí 0: `10`.
- Vị trí 1: `10+0=10`.
- Vị trí 2: `10+0+20=30`.
- ...

**Tại sao đúng?** Vì các chênh lệch `(a[j]-a[j-1])` ở giữa tự **triệt tiêu** khi cộng.

In [6]:
running = 0
recovered = []
for step in diff_from_a:
    running += step
    recovered.append(running)
    print(f'độ thay đổi={step:>3}, tổng tới đây={running:>3}')
assert recovered == a

độ thay đổi= 10, tổng tới đây= 10
độ thay đổi=  0, tổng tới đây= 10
độ thay đổi= 20, tổng tới đây= 30
độ thay đổi=  0, tổng tới đây= 30
độ thay đổi=-10, tổng tới đây= 20


## 8. Bước đột phá — cập nhật cả đoạn bằng **2 điểm mốc**

Muốn **cộng `v` vào tất cả phần tử** trong đoạn `[left, right]` (index 0-based, hai đầu đều thuộc đoạn), ta không cần chạm vào mọi phần tử.

Chỉ cần cập nhật `diff`:

1. `diff[left] += v`: từ vị trí `left` trở đi, các tổng tiền tố đều tăng `v`.
2. `diff[right+1] -= v`: bắt đầu từ vị trí **sau** `right`, hủy mức tăng đó.

Nếu `right` là vị trí cuối mảng, mốc hủy nằm ở phần tử **phụ `diff[n]`**, không có trong kết quả.

In [7]:
n_demo = 6
d = [0] * (n_demo + 1)
left, right, v = 1, 3, 5  # index: cập nhật các vị trí 1, 2, 3.
d[left] += v
d[right + 1] -= v
print('Sau 2 mốc thay đổi:', d)
result_demo = []
total = 0
for step in d[:n_demo]:
    total += step
    result_demo.append(total)
print('Cộng dồn để thấy cả đoạn:', result_demo)
assert result_demo == [0,5,5,5,0,0]

Sau 2 mốc thay đổi: [0, 5, 0, 0, -5, 0, 0]
Cộng dồn để thấy cả đoạn: [0, 5, 5, 5, 0, 0]


### 8.1. Quy đổi về số chuyến bay 1-based

Booking `[first, last, seats]` đánh số chuyến từ **1**. Trong Python:

- Điểm bắt đầu có index `first - 1`.
- Điểm ngay **sau** chuyến `last` có index `last`.

Vì vậy, đúng **hai lệnh**:

```python
diff[first - 1] += seats
diff[last] -= seats
```

Khởi tạo `diff = [0] * (n+1)` để `diff[last]` **luôn hợp lệ**, kể cả `last == n`. Phần tử thứ `n+1` chỉ là *sentinel* (mốc phụ) và bị bỏ qua khi tạo đáp án dài `n`.

In [8]:
d = [0] * 6
for first, last, seats in bookings_example:
    d[first - 1] += seats
    d[last] -= seats
    print(f'Sau booking [{first},{last},{seats}]: {d}')
assert d == [10, 45, -10, -20, 0, -25]

Sau booking [1,2,10]: [10, 0, -10, 0, 0, 0]
Sau booking [2,3,20]: [10, 20, -10, -20, 0, 0]
Sau booking [2,5,25]: [10, 45, -10, -20, 0, -25]


## 9. Trace thuật toán tối ưu với ví dụ chính thức

`bookings=[[1,2,10],[2,3,20],[2,5,25]]`, `n=5`.

| Bước | Mốc bắt đầu | Mốc hủy | `diff` dài 6 |
|---|---|---|---|
| Khởi tạo | — | — | `[0,0,0,0,0,0]` |
| Booking 1 | `diff[0]+=10` | `diff[2]-=10` | `[10,0,-10,0,0,0]` |
| Booking 2 | `diff[1]+=20` | `diff[3]-=20` | `[10,20,-10,-20,0,0]` |
| Booking 3 | `diff[1]+=25` | `diff[5]-=25` | `[10,45,-10,-20,0,-25]` |

Cộng dồn 5 phần tử đầu: **`[10,55,45,25,25]`**.

**Lưu ý:** giá trị âm trong `diff` hoàn toàn bình thường: nó là một mức **giảm thay đổi**, không có nghĩa số ghế thực tế âm.

In [9]:
def trace_flight_bookings(bookings, n):
    diff = [0] * (n + 1)
    print('Ban đầu:', diff)
    for first, last, seats in bookings:
        diff[first - 1] += seats
        diff[last] -= seats
        print(f'Booking {first}..{last}, +{seats}: diff={diff}')
    answer = []
    running = 0
    for i in range(n):
        running += diff[i]
        answer.append(running)
        print(f'Chuyến {i+1}: running={running}')
    return answer

assert trace_flight_bookings(bookings_example, 5) == [10,55,45,25,25]

Ban đầu: [0, 0, 0, 0, 0, 0]
Booking 1..2, +10: diff=[10, 0, -10, 0, 0, 0]
Booking 2..3, +20: diff=[10, 20, -10, -20, 0, 0]
Booking 2..5, +25: diff=[10, 45, -10, -20, 0, -25]
Chuyến 1: running=10
Chuyến 2: running=55
Chuyến 3: running=45
Chuyến 4: running=25
Chuyến 5: running=25


## 10. Code tối ưu — hàm độc lập dễ đọc

Ta đã có đủ các mảnh: `diff` dài `n+1`, 2 cập nhật cho mỗi booking, rồi một vòng `for` cộng dồn.

In [10]:
def corp_flight_bookings_diff(bookings, n):
    diff = [0] * (n + 1)
    for first, last, seats in bookings:
        diff[first - 1] += seats
        diff[last] -= seats
    answer = []
    current = 0
    for i in range(n):
        current += diff[i]
        answer.append(current)
    return answer

print(corp_flight_bookings_diff(bookings_example, 5))
assert corp_flight_bookings_diff(bookings_example, 5) == [10,55,45,25,25]

[10, 55, 45, 25, 25]


## 11. Phiên bản LeetCode — đúng chữ ký `corpFlightBookings`

Trong bài nộp, không in từng bước. Dùng list có `n+1` phần tử để tránh nhánh điều kiện ở cuối đoạn.

In [11]:
from typing import List

class Solution:
    def corpFlightBookings(self, bookings: List[List[int]], n: int) -> List[int]:
        diff = [0] * (n + 1)
        for first, last, seats in bookings:
            diff[first - 1] += seats
            diff[last] -= seats
        running = 0
        for i in range(n):
            running += diff[i]
            diff[i] = running
        return diff[:n]

assert Solution().corpFlightBookings(bookings_example, 5) == [10,55,45,25,25]
assert Solution().corpFlightBookings([[1,2,10],[2,2,15]], 2) == [10,25]
print('PASS: 2 ví dụ chính thức')

PASS: 2 ví dụ chính thức


## 12. Vì sao thuật toán đúng? — chứng minh bằng đóng góp của từng booking

Xét **một booking** `(first, last, seats)`.

Sau khi thêm `+seats` ở `first-1` và `−seats` ở `last`, tổng tiền tố tại index `i` thay đổi:

- `i < first-1`: chưa gặp mốc tăng → thêm `0` ghế.
- `first-1 <= i < last`: đã gặp mốc tăng nhưng chưa gặp mốc hủy → thêm `seats` ghế.
- `i >= last`: gặp **cả hai mốc** → `+seats-seats=0`.

Vậy booking chỉ đóng góp cho **đúng đoạn `first..last`**. Do phép cộng có tính giao hoán/kết hợp, tổng đóng góp của mọi booking là tổng số ghế cần trả về.

## 13. Time / Space Complexity

Gọi `m = len(bookings)` và `n` là số chuyến bay.

| Phương pháp | Thời gian | Bộ nhớ bổ sung |
|---|---|---|
| Brute Force: duyệt từng chuyến trong mỗi booking | `O(m*n)` worst-case | `O(n)` cho đáp án |
| Difference Array + Prefix Sum | **`O(m+n)`** | **`O(n)`** |

Mỗi booking chỉ tạo **hai cập nhật O(1)**, nên xử lý `m` booking mất `O(m)`. Cộng dồn kết quả `n` chuyến mất `O(n)`.

**Không nhầm:** bản tối ưu không phải `O(1)` cho toàn bài: ta vẫn phải trả về mảng dài `n`.

## 14. Một cách viết khác — `itertools.accumulate`

Thư viện chuẩn có `accumulate`, trả lần lượt tổng tiền tố. Dễ viết gọn nhưng cần hiểu *bản chất cộng dồn* trước; code nộp phía trên không cần import phụ.

In [12]:
from itertools import accumulate

def corp_flight_bookings_accumulate(bookings, n):
    diff = [0] * (n + 1)
    for first, last, seats in bookings:
        diff[first - 1] += seats
        diff[last] -= seats
    return list(accumulate(diff[:n]))

assert corp_flight_bookings_accumulate(bookings_example, 5) == [10,55,45,25,25]
print('PASS: accumulate đúng như cộng dồn bằng for')

PASS: accumulate đúng như cộng dồn bằng for


## 15. Kiểm thử edge cases

Các trường hợp dễ gây lỗi:
- `n=1`, booking phủ đúng một chuyến.
- `last=n`, mốc giảm nằm ở `diff[n]`, không được đưa vào đáp án.
- `first=last`: đặt ghế cho đúng **một** chuyến.
- Nhiều booking chồng lấp lên nhau.
- Có chuyến không được đặt ghế, kết quả phải là `0`.

In [13]:
fixed_cases = [
    ([[1,1,7]], 1, [7]),
    ([[1,3,4]], 3, [4,4,4]),
    ([[2,2,6]], 4, [0,6,0,0]),
    ([[1,2,10],[2,2,15]], 2, [10,25]),
    ([[2,4,1],[1,3,2],[3,3,10]], 5, [2,3,13,1,0]),
]
for bookings, n, expected in fixed_cases:
    actual = Solution().corpFlightBookings(bookings, n)
    assert actual == expected, (bookings, actual, expected)
print('PASS:', len(fixed_cases), 'edge cases')

PASS: 5 edge cases


### 15.1. Random test: đối chiếu với Brute Force

Ta dùng số ngẫu nhiên có **seed cố định**, nên kết quả chạy lại trên máy khác giống nhau. Với đầu vào nhỏ, brute force là *oracle* độc lập để đối chiếu.

In [14]:
from random import Random
rng = Random(1109)
for _ in range(400):
    n = rng.randint(1, 25)
    m = rng.randint(1, 25)
    bookings = []
    for _ in range(m):
        left = rng.randint(1, n)
        right = rng.randint(left, n)
        seats = rng.randint(1, 20)
        bookings.append([left, right, seats])
    expect = corp_flight_bookings_bruteforce(bookings, n)
    actual = Solution().corpFlightBookings(bookings, n)
    alt = corp_flight_bookings_accumulate(bookings, n)
    assert actual == alt == expect
print('PASS: 400 test ngẫu nhiên, 3 cách cho cùng kết quả')

PASS: 400 test ngẫu nhiên, 3 cách cho cùng kết quả


### 15.2. Stress test ở kích thước lớn nhất của đề

Không dùng brute force cho trường hợp `m=n=20.000`: nó có thể cần hàng trăm triệu cập nhật. Ta kiểm tra tính chất đơn giản có đáp án biết trước: tất cả booking đều đặt một ghế trên toàn bộ các chuyến → mỗi chuyến có `m` ghế.

In [15]:
n_max = 20_000
bookings_max = [[1, n_max, 1] for _ in range(n_max)]
large_ans = Solution().corpFlightBookings(bookings_max, n_max)
assert len(large_ans) == n_max
assert all(x == n_max for x in large_ans)
print('PASS: 20.000 chuyến, 20.000 booking; mỗi chuyến có', large_ans[0], 'ghế')

PASS: 20.000 chuyến, 20.000 booking; mỗi chuyến có 20000 ghế


## 16. Lỗi người mới thường gặp

**Lỗi 1:** dùng `diff[first] += seats` thay vì `diff[first-1]`, làm lệch sang phải một chuyến.

**Lỗi 2:** dùng `diff[last-1] -= seats` để hủy, khiến chuyến cuối **không được cộng**.

**Lỗi 3:** `diff=[0]*n` mà vẫn cập nhật `diff[last]` khi `last=n`, gây `IndexError`.

**Lỗi 4:** trả `diff` ngay sau các cập nhật. `diff` là mảng **thay đổi**, phải lấy tổng tiền tố để ra số ghế.

**Lỗi 5:** trả luôn `n+1` phần tử. Đáp án chỉ có **`n` chuyến bay**.

In [16]:
# Minh họa lỗi 2: hủy ghế quá sớm.
wrong = [0] * 4
wrong[1-1] += 5
wrong[2-1] -= 5  # Sai: phải trừ ở index 2, không phải index 1!
good = [0] * 4
good[1-1] += 5
good[2] -= 5

def get_prefix_first_n(diff, n):
    out, running = [], 0
    for x in diff[:n]:
        running += x
        out.append(running)
    return out

print('Sai:', get_prefix_first_n(wrong, 3))
print('Đúng:', get_prefix_first_n(good, 3))
assert get_prefix_first_n(wrong, 3) == [5,0,0]
assert get_prefix_first_n(good, 3) == [5,5,0]

Sai: [5, 0, 0]
Đúng: [5, 5, 0]


## 17. Nhận diện pattern — range addition + point query (offline)

Mảng hiệu phù hợp khi:
- Có **nhiều thao tác cộng cùng một giá trị lên cả đoạn liên tiếp**.
- **Không cần lấy kết quả sau mỗi thao tác**; ta có thể nhận tất cả booking trước rồi tính một lần cuối.
- Kết quả cần cho **tất cả vị trí** hoặc từng vị trí sau khi hoàn tất cập nhật.

Đây là kỹ thuật **offline range updates**. Nếu bài yêu cầu hỏi kết quả xen kẽ lúc đang cập nhật, bạn có thể cần Fenwick Tree hoặc Segment Tree; không nên áp dụng mảng hiệu tĩnh một cách máy móc.

## 18. Biến thể — cập nhật một đoạn tùy ý trên mảng có sẵn

Nếu mảng gốc không phải toàn số 0 mà là `a=[2,5,1,6]`, ta có thể:

1. Tính mảng hiệu của `a`.
2. Cập nhật mốc `+v` và `−v` cho đoạn `[l,r]`.
3. Tính prefix sum để lấy mảng mới.

Đây là cùng một nguyên lý, áp dụng được cho nhiệt độ, số lượng khách, số đơn hàng theo ngày, ...

In [17]:
def range_add_on_existing_array(values, updates):
    n = len(values)
    diff = [0] * (n + 1)
    diff[0] = values[0]
    for i in range(1, n):
        diff[i] = values[i] - values[i - 1]
    for left, right, add in updates:  # index 0-based.
        diff[left] += add
        diff[right + 1] -= add
    return get_prefix_first_n(diff, n)

assert range_add_on_existing_array([2,5,1,6], [(1,2,4)]) == [2,9,5,6]
print(range_add_on_existing_array([2,5,1,6], [(1,2,4)]))

[2, 9, 5, 6]


## 19. Bài tập tự luyện — 5 mức

**Bài 1 (tính tay):** với `bookings=[[1,3,5],[2,2,4]], n=3`, tính `answer`.

**Bài 2 (đoán output):** tính tổng tiền tố của `[3,-1,0,5,-2]`.

**Bài 3 (TODO):** viết `exercise_prefix_sum(diff)` không dùng `itertools`.

**Bài 4 (TODO):** viết `exercise_bookings(bookings,n)` bằng mảng hiệu, không gọi sẵn `corp_flight_bookings_diff`.

**Bài 5 (biến thể):** viết `exercise_range_add(values,updates)` cho `updates=[(left,right,add),...]` với **chỉ số 0-based**, giữ nguyên mảng đầu vào.

In [18]:
def exercise_prefix_sum(diff):
    # TODO: tạo running=0, cộng từng phần tử và append vào kết quả.
    pass

def exercise_bookings(bookings, n):
    # TODO: diff có n+1 phần tử; cập nhật hai mốc; lấy prefix đầu n.
    pass

def exercise_range_add(values, updates):
    # TODO: tính diff từ values gốc rồi thêm mốc cho từng update.
    pass

# Tự làm xong mới mở comment:
# assert exercise_prefix_sum([3,-1,0,5,-2]) == [3,2,2,7,5]
# assert exercise_bookings([[1,3,5],[2,2,4]], 3) == [5,9,5]
# assert exercise_range_add([2,5,1,6],[(1,2,4)]) == [2,9,5,6]

## 20. Gợi ý ba mức

| Bài | Mức 1 — ý tưởng | Mức 2 — cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | Booking thứ hai chỉ cộng ở chuyến 2 | `5+4=9` | `[5,9,5]` |
| 2 | Cộng dồn từ trái qua phải | `running += x` | `[3,2,2,7,5]` |
| 3 | `running` nhớ tổng tới hiện tại | `out.append(running)` | `for x in diff: running+=x; out.append(running)` |
| 4 | Chỉ cập nhật 2 mốc cho mỗi booking | `diff[first-1]`, `diff[last]` | `return prefix(diff)[:n]` |
| 5 | Giá trị ban đầu cũng đổi được thành diff | `diff[i]=values[i]-values[i-1]` | `diff[left]+=v; diff[right+1]-=v` |

## 21. Đáp án tham khảo — sau bài tự luyện

**Bài 1:** `[5,9,5]`. **Bài 2:** `[3,2,2,7,5]`.

In [19]:
def answer_prefix_sum(diff):
    running = 0
    out = []
    for step in diff:
        running += step
        out.append(running)
    return out

def answer_bookings(bookings, n):
    d = [0] * (n + 1)
    for first, last, seats in bookings:
        d[first - 1] += seats
        d[last] -= seats
    return answer_prefix_sum(d)[:n]

def answer_range_add(values, updates):
    d = [0] * (len(values) + 1)
    d[0] = values[0]
    for i in range(1, len(values)):
        d[i] = values[i] - values[i - 1]
    for left, right, increment in updates:
        d[left] += increment
        d[right + 1] -= increment
    return answer_prefix_sum(d)[:len(values)]

assert answer_prefix_sum([3,-1,0,5,-2]) == [3,2,2,7,5]
assert answer_bookings([[1,3,5],[2,2,4]], 3) == [5,9,5]
x = [2,5,1,6]
assert answer_range_add(x, [(1,2,4)]) == [2,9,5,6]
assert x == [2,5,1,6]  # Không sửa đầu vào.
print('PASS: đáp án 3 bài code')

PASS: đáp án 3 bài code


## 22. Template nhớ nhanh + checklist submit

```python
diff = [0] * (n+1)
for first, last, value in bookings:
    diff[first-1] += value
    diff[last] -= value
# Sau đó cộng dồn n vị trí đầu.
```

- [ ] Tôi phân biệt **số hiệu 1-based** và **list index 0-based**.
- [ ] Tôi hiểu vì sao cập nhật `diff[last]` chứ không phải `diff[last-1]`.
- [ ] Tôi biết `diff` chưa phải số ghế thật và phải **lấy tổng tiền tố**.
- [ ] Tôi chỉ trả `n` phần tử, test `last=n` và `first=last`.
- [ ] Complexity: `O(m+n)` thời gian, `O(n)` bộ nhớ.

**Câu chốt:** cập nhật một đoạn bằng **2 điểm mốc**, rồi dùng prefix sum biến các mốc thành tác động trên toàn đoạn.